In [1]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
import json
import numpy as np
from datetime import datetime

from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_community.chat_models import ChatOllama


# Path

In [2]:
print(os.path.exists(r"C:\Users\vaish\AIML.pdf"))

True


In [3]:
DATA_PATH = "data/"
DB_PATH = "vectorstore/"
MEMORY_PATH = "memory.json"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(DB_PATH, exist_ok=True)

# Document Ingestion (PDF + WEB)

converting raw documents --> LanggChain Documents

In [4]:
def load_documents(pdf_paths=[], urls=[]):
    docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs.extend(loader.load())

    for url in urls:
        loader = WebBaseLoader(url)
        docs.extend(loader.load())

    return docs

# Text Chunking 

breaks documents into chunks for embedding 

In [5]:
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    return splitter.split_documents(docs)

# embeddings + FAISS

Convert text --> vectors 
store locally


In [6]:
def create_vectorstore(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    clean_chunks = []
    for doc in chunks:
        text = doc.page_content.strip()

        # ❌ REMOVE BAD CHUNKS
        if (
            len(text) > 80 and
            not text.isdigit() and
            len(text.split()) > 5
        ):
            clean_chunks.append(doc)

    print(f"✅ Clean chunks kept: {len(clean_chunks)}")

    vectorstore = FAISS.from_documents(clean_chunks, embeddings)
    vectorstore.save_local(DB_PATH)

    return vectorstore

# Retriever Setup

Enables Semantic Search

In [7]:
def load_vectorstore():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return FAISS.load_local(
        DB_PATH,
        embeddings,
        allow_dangerous_deserialization=True  
    )

# LLM Setup

local interface with LLaMA 3

In [8]:
llm = ChatOllama(model="phi")

C:\Users\vaish\AppData\Local\Temp\ipykernel_39512\1496863374.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="phi")


# Memory System (SHORT + LONG TERM)

Enables follow ups + personalization

In [9]:
def load_memory():
    if os.path.exists(MEMORY_PATH):
        with open(MEMORY_PATH, "r") as f:
            return json.load(f)
    return []

def save_memory(memory):
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2)

def update_memory(query, response):
    memory = load_memory()
    memory.append({
        "query": query,
        "response": response,
        "time": str(datetime.now())
    })
    save_memory(memory)
    

# Query Classifier

Routes queries to proper pipeline

In [10]:
def classify_query(query):
    prompt = f"""
    Classify this query:
    {query}

    Categories:
    - factual
    - analytical
    - simple

    Only return category.
    """
    return llm.invoke(prompt).content.strip().lower()

# Multi-Query RAG

Improves recall and reduce hallucination

In [11]:
def generate_queries(query):
    prompt = F"""
    Generate 3 alternative search queries for:
    {query}
    """
    result = llm.invoke(prompt).content
    
    return list(set(result.split("\n")))

Retrieval Merge

In [12]:
def multi_query_retrieval(query, retriever):
    queries = generate_queries(query)
    docs = []

    for q in queries:
        docs.extend(retriever.invoke(q))   

    # Deduplicate
    unique = {doc.page_content: doc for doc in docs}
    return list(unique.values())

# Multi-Agent System

Planner Agent, 
Retriver Agent,
Analyst Agent,
Critic Agent.

In [13]:
# Planner Agent

def planner_agent(query):
    return f"Plan: Break query '{query}' into steps."

In [14]:
# Retriever Agent

def retriever_agent(query, retriever):
    return multi_query_retrieval(query, retriever)

In [15]:
def analyst_agent(query, docs):
    context = "\n".join([d.page_content for d in docs[:5]])

    prompt = f"""
You are a strict AI assistant.

Answer ONLY from the given context.

RULES:
- If answer is NOT in context → say: "Answer not found in provided documents."
- Do NOT include hashtags
- Do NOT include extra questions
- Keep answer clean and structured

Context:
{context}

Question:
{query}

Answer:
"""

    return llm.invoke(prompt).content

In [16]:
def critic_agent(answer, docs):
    context = " ".join([d.page_content for d in docs[:3]])

    prompt = f"""
Check if the answer is supported by context.

Return ONLY:
- Valid
- Invalid

Answer:
{answer}

Context:
{context}
"""

    return llm.invoke(prompt).content.strip()

# Evaluation Layer

In [17]:
def evaluate(answer, docs):
    relevance = len(docs) / 10
    confidence = min(1.0, relevance)

    return {
        "relevance": relevance,
        "confidence": confidence
    }

In [18]:
#Format

In [19]:
def clean_output(text):
    text = text.replace("```", "")
    text = text.replace("python", "")

    # Remove unwanted trailing question
    if "Question:" in text:
        text = text.split("Question:")[0]

    # Remove hashtags
    text = text.replace("#", "")

    return text.strip()

In [20]:
def format_output(result):
    print("\n🧠 ANSWER:\n")
    print(result["answer"])

    print("\n📊 CONFIDENCE:", result["confidence"])
    print("✅ VALIDATION:", result["validation"])

    print("\n📚 SOURCES:")
    for i, doc in enumerate(result["sources"]):
        print(f"{i+1}. Page {doc.metadata.get('page')}")

# Final Pipeline

In [21]:
def run_pipeline(query):

    # ✅ FIX: Create vector DB if not exists
    if not os.path.exists(os.path.join(DB_PATH, "index.faiss")):
        print("⚠️ Vector DB not found. Creating now...")

        docs = load_documents(
            pdf_paths=[r"data/AIML.pdf"],
            urls=[]
        )

        chunks = split_documents(docs)
        create_vectorstore(chunks)

        print("✅ Vector DB created successfully.")

    # ✅ Load vector DB
    vectorstore = load_vectorstore()

    # 🔥 IMPROVED RETRIEVER
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    query_type = classify_query(query)

    docs = retriever_agent(query, retriever)

    # 🔍 ✅ DEBUG BLOCK (ADD THIS HERE)
    print("\n🔍 Retrieved Docs Preview:\n")
    for i, d in enumerate(docs[:3]):
        print(f"\n--- DOC {i+1} ---\n", d.page_content[:200])

    # 🔥 Continue pipeline
    answer = analyst_agent(query, docs)
    validation = critic_agent(answer, docs)
    metrics = evaluate(answer, docs)

    update_memory(query, answer)

    return {
        "answer": clean_output(answer),
        "sources": docs[:3],
        "validation": validation.strip(),
        "confidence": metrics["confidence"]
    }

# Test Query

In [23]:
run_pipeline("Explain what is Artificial Intelligence")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔍 Retrieved Docs Preview:


--- DOC 1 ---
 49

--- DOC 2 ---
 41

--- DOC 3 ---
 48


{'answer': 'Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence. This includes tasks such as visual perception, speech recognition, decision making, and problem-solving. AI systems are designed to learn from data, adapt to new information, and improve their performance over time. They can analyze complex patterns, make predictions, and provide recommendations based on the input they receive.\n\nAI is a multidisciplinary field that combines computer science, mathematics, statistics, and cognitive psychology. It involves developing algorithms, models, and techniques that allow machines to simulate human-like intelligence. This includes machine learning, natural language processing, computer vision, and robotics.\n\nThe goal of AI is to create systems that can perform tasks more efficiently and effectively than humans, while also providing the ability to learn from experience and adapt to changing circ